In [62]:
import numpy as np
import torch 
import itertools
import pandas as pd
import matplotlib.pyplot as plt
from anastruct import SystemElements
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt 
import openpyxl as pxl
import torch.nn.functional as F
import os
import re
import csv
from datetime import datetime
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics  import mean_absolute_error, r2_score


In [63]:
#General Parameters for improve results
"""

"""

seed=100
alpha =0.4
dropout = 0.0
Lambda =0.1
epochs = 500
patience =50
batch_size=16

In [64]:
#import Excel Data

notebook_dir = os.getcwd()
file_path =os.path.join(notebook_dir,'Data','MLPARTA.xlsx')

In [65]:
#read Excel with formulas using Openpyxl

wb = pxl.load_workbook(file_path)
ws = wb.active 

headers = [ws.cell(1,col).value for col in range (1,ws.max_column+1)]

#Convert excel language to python


letter_to_name = {}
for col in range(1,ws.max_column+1):
    letter = pxl.utils.get_column_letter(col)
    letter_to_name[letter] =headers[col-1]

def excel_to_python(formula, letter_to_name):
    result = formula
    result = result.lstrip('=')
    result = result.replace('$', '')
    result = re.sub(r'\bPI\(\)', 'np.pi', result)

    def replace_ref(match):
        letter = match.group(1)
        if letter in letter_to_name:
            name = letter_to_name[letter]
            return f"df['{name}']"
        return match.group(0)

    result = re.sub(r'\b([A-Z]{1,3})(\d+)\b', replace_ref, result)
    
    excel_to_np = {
        'SIN'   : 'np.sin',
        'COS'   : 'np.cos',
        'TAN'   : 'np.tan',
        'ATAN'  : 'np.arctan',
        'ATAN2' : 'np.arctan2',
        'ASIN'  : 'np.arcsin',
        'ACOS'  : 'np.arccos',
        'SQRT'  : 'np.sqrt',
        'ABS'   : 'np.abs',
        'EXP'   : 'np.exp',
        'LOG'   : 'np.log',
        'LOG10' : 'np.log10',
        'POWER' : 'np.power',
        'MOD'   : 'np.mod',
        'FLOOR' : 'np.floor',
        'CEIL'  : 'np.ceil',
        'ROUND' : 'np.round',
        'MAX'   : 'np.maximum',
        'MIN'   : 'np.minimum',
        'SUM'   : 'np.sum',
    }

    for excel_fn, np_fn in excel_to_np.items():
        result = re.sub(rf'\b{excel_fn}\b', np_fn, result)

    result = result.replace('^', '**')
    result = result.lstrip('=')

    return result


# Find Formula
formula_list = {}
for col in range(1,ws.max_column+1):
    cell =ws.cell(2,col)
    value = cell.value
    name = headers[col-1]
    if isinstance(value,str) and value.startswith('='):
        formula_list[name]=excel_to_python(value,letter_to_name)

raw_cols = [h for h in headers if h not in formula_list]

def compute_features(df,formula_list):
    df = df.copy()
    remaining = dict(formula_list)
    max_passes = len(formula_list)+1
    passes =0 

    while remaining and passes < max_passes:
        passes +=1
        newly_computed=[]
        
        for col_name, expr in remaining.items():
            try:
                df[col_name]=eval(expr)
                newly_computed.append(col_name)
            except Exception:
                pass
        for col in newly_computed:
            del remaining[col]
        if not newly_computed:
            print(f"\n Warning Error Newly Computed")
            break
    return df

wb_data = pxl.load_workbook(file_path, data_only=True)
ws_data = wb_data.active
data    = [row for row in ws_data.iter_rows(min_row=2, values_only=True)]
df_raw  = pd.DataFrame(data, columns=headers)
df_computed = compute_features(df_raw[raw_cols], formula_list)
df_training = df_computed.drop(columns=[None], errors='ignore').dropna().reset_index(drop=True)
print(f"df_training shape: {df_training.shape}")   # should be (100, 28)


df_training shape: (100, 28)


In [66]:
# # Verify 
# # ── 8. Verify against Excel ───────────────────────────────────
# print(f"\n{'='*60}")
# print(f"  Verify — Python vs Excel (row 1)")
# print(f"{'='*60}")
# print(f"{'Column':>15} | {'Excel':>12} | {'Python':>12} | {'Match':>6}")
# print(f"{'-'*60}")

# match_count = 0
# for col in formula_list.keys():
#     if col in df_raw.columns and col in df_computed.columns:
#         excel_val  = df_raw[col].iloc[0]
#         python_val = df_computed[col].iloc[0]
#         if excel_val is not None and not pd.isna(excel_val):
#             match = abs(excel_val - python_val) < 1e-4
#             if match:
#                 match_count += 1
#             print(f"{col:>15} | {excel_val:>12.4f} | {python_val:>12.4f} | "
#                   f"{'✓' if match else '✗':>6}")

# print(f"\n  Matched : {match_count} / {len(formula_list)} columns")
# print(f"{'='*60}")

# # ── 9. Final dataset summary ──────────────────────────────────
# print(f"\n{'='*60}")
# print(f"  FINAL DATASET")
# print(f"{'='*60}")
# print(f"  Shape   : {df_computed.shape}")
# print(f"  Columns : {df_computed.columns.tolist()}")
# print(f"\n{'='*60}")
# print(f"  STATISTICS")
# print(f"{'='*60}")
# print(f"{'Column':>15} | {'Min':>12} | {'Max':>12} | {'Mean':>12}")
# print(f"{'-'*60}")
# for col in df_computed.columns:
#     vals = df_computed[col].dropna()
#     if vals.dtype in [float, int]:
#         print(f"{col:>15} | {vals.min():>12.4f} | "
#               f"{vals.max():>12.4f} | {vals.mean():>12.4f}")
# print(f"{'='*60}")

In [67]:
# # Print formulas 
# print(f"  Formula — {len(formula_list)} columns")
# for col_name, expr in formula_list.items():
#     print(f"\n  Column  : {col_name}")
#     print(f"  Python  : {expr}")


In [68]:
# Define Target Columns 

target_col = ['Ftan_c','Power']
n_output = len(target_col) #save size 
constant_cols={}

def is_constant(series, cv_threshold=0.01):
    std  = series.std()
    mean = series.mean()
    if pd.isna(std) or std == 0:    return True
    if mean == 0 or pd.isna(mean):  return std < 1e-10
    return (std / abs(mean)) < cv_threshold

for col in df_training.columns:
    if col is None or col in target_col:
        continue
    series = df_training[col].dropna()   # guard against empty columns
    if len(series) == 0:
        continue
    if is_constant(series):
        constant_cols[col] = series.iloc[0]

C = df_training[list(constant_cols.keys())].copy()
Y = df_training[target_col].copy()
X = df_training.drop(columns=list(constant_cols.keys()) + target_col, errors='ignore').copy()
X = X[[col for col in X.columns if col is not None]]

feature_cols = list(X.columns)  

x = torch.tensor(X.values, dtype=torch.float32)
y = torch.tensor(Y.values, dtype=torch.float32)
c = torch.tensor(C.values, dtype=torch.float32)

print(f"  Dataframe check ")

print(f"\n  C — Constants : {C.shape}")
print(f"    {C.columns.tolist()}")
print(f"\n  Y — Target    : {Y.shape}")
print(f"    {Y.columns.tolist()}")
print(f"\n  X — Features  : {X.shape}")
print(f"    {X.columns.tolist()}")

print(f"  x tensor : {x.shape}")
print(f"  y tensor : {y.shape}")
print(f"  c tensor : {c.shape}")
print(f"  NaN in x : {torch.isnan(x).sum().item()}")
print(f"  NaN in y : {torch.isnan(y).sum().item()}")
print(f"  NaN in c : {torch.isnan(c).sum().item()}")


  Dataframe check 

  C — Constants : (100, 5)
    ['F_PB', 'Density', 'W_motor', 'Fx_s', 'Ftotal_s']

  Y — Target    : (100, 2)
    ['Ftan_c', 'Power']

  X — Features  : (100, 21)
    ['R', 'L', 'offset_e', 'cross_area', 'ac_rad', 'm_crank', 'm_link', 'm_tot', 'm_slid', 'slider_y', 'a_cl', 's_omega', 's_alpha', 's_inertia', 'F_cl', 'Fx_cl', 'Fy_cl', 'c_inertia', 'Torque', 'Beta', 'QRR']
  x tensor : torch.Size([100, 21])
  y tensor : torch.Size([100, 2])
  c tensor : torch.Size([100, 5])
  NaN in x : 0
  NaN in y : 0
  NaN in c : 0


In [69]:
#Parameters

batch_size =16
num_workers=0   # cpu (AI Recommended)

class Training(Dataset):

    def __init__(self, x_mix, y_mix, x_orig):
        assert x_mix.shape[0] == y_mix.shape[0] == x_orig.shape[0]
        self.x_mix  = x_mix
        self.y_mix  = y_mix
        self.x_orig = x_orig

    def __len__(self):return self.x_mix.shape[0]
    def __getitem__(self, idx):return (self.x_mix[idx],self.y_mix[idx],self.x_orig[idx],)

class Validation(Dataset):

    def __init__(self,x,y,c):
        assert x.shape[0] == y.shape[0] == c.shape[0]
        self.x = x
        self.y = y
        self.c = c

    def __len__(self):return self.x.shape[0]
    def __getitem__(self, idx): return self.x[idx], self.y[idx], self.c[idx]

def get_dataloaders(train_ds,val_ds,batch_size,num_workers,seed) -> tuple[DataLoader, DataLoader]:
    g = torch.Generator().manual_seed(seed) #shuffle generator

    train_loader = DataLoader(
        train_ds,
        batch_size  = batch_size,
        shuffle     = True,
        drop_last   = True,
        num_workers = num_workers,
        generator   = g, #shuffle
    )
    val_loader = DataLoader(
        val_ds,
        batch_size  = batch_size,
        shuffle     = False,
        drop_last   = False,
        num_workers = num_workers,
    )
    return train_loader, val_loader


In [70]:
#  Training, Test and Validation DataFrames

def split_data(x, y, c, seed):
    torch.manual_seed(seed)
    sorted_idx = torch.argsort(y[:, -1])
    train_idx, val_idx, test_idx = [], [], []
    
    for i, idx in enumerate(sorted_idx):
        # Using modulo 20 to allow for 5% increments
        r = i % 20
        
        if r < 15:    
            train_idx.append(idx.item())   # 15/20 = 75%
        elif r < 18:  
            val_idx.append(idx.item())     # 3/20  = 15%
        else:         
            test_idx.append(idx.item())    # 2/20  = 10%
            
    return (
        torch.tensor(train_idx),
        torch.tensor(val_idx),
        torch.tensor(test_idx),
    )

def mixup(x, y, c, alpha=alpha, seed=seed):
    np.random.seed(seed)
    N     = x.size(0)
    lam   = np.random.beta(alpha, alpha, size=N)
    lam   = np.maximum(lam, 1 - lam)
    lam_t = torch.tensor(lam, dtype=torch.float32)
    idx_p = torch.randperm(N)
    lam_x = lam_t.unsqueeze(1)
    lam_y = lam_x if y.dim() == 2 else lam_t
    x_mix = lam_x * x + (1 - lam_x) * x[idx_p]
    y_mix = lam_y * y + (1 - lam_y) * y[idx_p]
    c_mix = lam_x * c + (1 - lam_x) * c[idx_p]
    return x_mix, y_mix, c_mix


train_idx, val_idx, test_idx = split_data(x, y, c, seed=seed)

print(f'  Train : {len(train_idx):>3} samples  ({len(train_idx)/len(x)*100:.0f}%)')
print(f'  Val   : {len(val_idx):>3} samples  ({len(val_idx)/len(x)*100:.0f}%)')
print(f'  Test  : {len(test_idx):>3} samples  ({len(test_idx)/len(x)*100:.0f}%)')
for col_i, col in enumerate(target_col):
    print(f'  {col} range — '
          f'Train [{y[train_idx,col_i].min():.2f}, {y[train_idx,col_i].max():.2f}]  '
          f'Val [{y[val_idx,col_i].min():.2f}, {y[val_idx,col_i].max():.2f}]  '
          f'Test [{y[test_idx,col_i].min():.2f}, {y[test_idx,col_i].max():.2f}]')


  Train :  75 samples  (75%)
  Val   :  15 samples  (15%)
  Test  :  10 samples  (10%)
  Ftan_c range — Train [52.46, 141.13]  Val [83.59, 138.74]  Test [64.74, 131.72]
  Power range — Train [10.77, 71.79]  Val [27.68, 73.40]  Test [28.60, 79.60]


In [71]:
class Sohoite(nn.Module):
    def __init__(self, input_dim, n_outputs=1, dropout=0.2,
                 hidden=[32, 64, 32]):              # ← add this
        super().__init__()
        def block(in_f, out_f):
            return nn.Sequential(
                nn.Linear(in_f, out_f),
                nn.Sigmoid(),
                nn.LayerNorm(out_f),
                nn.Sigmoid(),
                nn.Dropout(dropout),
            )
        self.block1 = block(input_dim,  hidden[0])  # ← use hidden
        self.block2 = block(hidden[0],  hidden[1])  # ← use hidden
        self.block3 = block(hidden[1],  hidden[2])  # ← use hidden
        self.head   = nn.Linear(hidden[2], n_outputs)
        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return self.head(x)
    
""""
Notes from PyTorch Documentation and Class 
- Linear is the based feature of any code 
- LayerNorm: normalizes the values, and works better for small batches but also of any sizes. It is slow compared to BatchNorm 
- Sigmoid: This is the activation function recommended by Dr. Ince
- Dropout: is a strategy to disable a neuron during training for reductancy. It can also be set at zero

Before starting, the code has to initialize the weights. Based on PyTorch Documentation recommends to use Xavier rather than nn.Linear since it was developed for Sigmoid
""" 

'"\nNotes from PyTorch Documentation and Class \n- Linear is the based feature of any code \n- LayerNorm: normalizes the values, and works better for small batches but also of any sizes. It is slow compared to BatchNorm \n- Sigmoid: This is the activation function recommended by Dr. Ince\n- Dropout: is a strategy to disable a neuron during training for reductancy. It can also be set at zero\n\nBefore starting, the code has to initialize the weights. Based on PyTorch Documentation recommends to use Xavier rather than nn.Linear since it was developed for Sigmoid\n'

In [72]:
def get_constant(name):
    if name not in constant_cols:
        raise KeyError(f"'{name}' not found")
    return constant_cols[name]

CHECK = ['F_cl', 'Torque', 'Power'] # Formulas Chosen to check => Can be Modified 
 
missing = [p for p in CHECK if p not in formula_list]
if missing:
    raise ValueError(f"CHECK missing from formula_list: {missing}")
def to_torch(expr):
    return expr.replace('np.', 'torch.')

exprs = {name: to_torch(formula_list[name])for name in CHECK}

constant_needed = [
    name for name in constant_cols 
    if any(f"df['{name}']" in expr for expr in exprs.values())
]

feat_cols = list(dict.fromkeys(match
    for expr in exprs.values()
    for match in re.findall(r"df\['(\w+)'\]", expr)
    if match in feature_cols
))

F_idx = {col: feature_cols.index(col) for col in feat_cols}  

 #weight of penalize for deviating from ground physics


def physics_loss(pred_norm, x_orig, x_mean, x_std, y_mean, y_std):
    df = {}

    # Find Features 
    for col, idx in F_idx.items():
        df[col] = x_orig[:, idx] * x_std[idx] + x_mean[idx]
    for name in CHECK:
        if name in feature_cols and name not in df:
            idx      = feature_cols.index(name)
            df[name] = x_orig[:, idx] * x_std[idx] + x_mean[idx]
    for name in constant_needed:
        df[name] = get_constant(name)

    # Include Target in formula─
    for name in target_col:
        if name not in df:
            i        = target_col.index(name)
            df[name] = (pred_norm[:, i]
                        * y_std[i].to(pred_norm.device)
                        + y_mean[i].to(pred_norm.device))

    # Define physics loss deviation
    total = torch.tensor(0.0, device=pred_norm.device)

    for name in CHECK:
        physics_val = eval(exprs[name])   # what the formula says it should be

        if name in target_col:
            i        = target_col.index(name)
            pred_val = (pred_norm[:, i]
                        * y_std[i].to(pred_norm.device)
                        + y_mean[i].to(pred_norm.device))
            total   += F.mse_loss(pred_val, physics_val.detach())
        else:
             total   += F.mse_loss(df[name], physics_val.detach())

    return Lambda *total

In [73]:
# Helper Functions 

def train_epoch(model, loader, optimizer, device, x_mean, x_std, y_mean, y_std):
    model.train()
    criterion  = nn.MSELoss()
    total_loss = 0.0
    for x_mix, y_mix, x_orig in loader:
        x_mix  = x_mix.to(device)
        y_mix  = y_mix.to(device)
        x_orig = x_orig.to(device)

        optimizer.zero_grad()

        pred = model(x_mix)

        loss_data    = criterion(pred, y_mix)
        loss_physics = physics_loss(
            pred, x_orig,
            x_mean.to(device), x_std.to(device),
            y_mean.to(device), y_std.to(device),
        )
        loss = loss_data + loss_physics  
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(x_mix)

    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, device):

    model.eval()
    criterion  = nn.MSELoss()
    total_loss = 0.0
    all_preds, all_actuals = [], []

    for x_batch, y_batch, _ in loader:  
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        pred = model(x_batch)
        loss = criterion(pred, y_batch)

        total_loss += loss.item() * len(x_batch)
        all_preds.append(pred.cpu())
        all_actuals.append(y_batch.cpu())

    return (
        total_loss / len(loader.dataset),
        torch.cat(all_preds),
        torch.cat(all_actuals),
    )
def tester(y_pred_norm, y_true_norm, y_mean, y_std,target_col):

    # Denormalize → Watts
    y_pred = (y_pred_norm * y_std + y_mean).numpy()
    y_true = (y_true_norm * y_std + y_mean).numpy()
    results={}
    for i, col in enumerate(target_col):
        err    = y_pred[:, i] - y_true[:, i]
        mae    = float(np.mean(np.abs(err)))
        rmse   = float(np.sqrt(np.mean(err ** 2)))
        ss_res = np.sum(err ** 2)
        ss_tot = np.sum((y_true[:, i] - y_true[:, i].mean()) ** 2)
        r2     = float(1 - ss_res / (ss_tot + 1e-8))
        results[col] = {'mae': mae, 'rmse': rmse, 'r2': r2}
    mean_mae = float(np.mean([v['mae'] for v in results.values()]))
    return results, mean_mae

In [74]:


device = torch.device(
    'mps'  if torch.backends.mps.is_available()  else
    'cuda' if torch.cuda.is_available()          else
    'cpu'
) # AI recommended 




In [75]:
# Set Up
x_tr  = x[train_idx];  y_tr  = y[train_idx]
x_val = x[val_idx];    y_val = y[val_idx]
x_te  = x[test_idx];   y_te  = y[test_idx]
c_val = c[val_idx];    c_te  = c[test_idx]

x_mean = x_tr.mean(dim=0)
x_std  = x_tr.std(dim=0).clamp(min=1e-8)
y_mean = y_tr.mean(dim=0)                  
y_std  = y_tr.std(dim=0).clamp(min=1e-8) 

x_tr_norm  = (x_tr  - x_mean) / x_std
x_val_norm = (x_val - x_mean) / x_std
x_te_norm  = (x_te  - x_mean) / x_std
y_tr_norm  = (y_tr  - y_mean) / y_std
y_val_norm = (y_val - y_mean) / y_std
y_te_norm  = (y_te  - y_mean) / y_std

#Mixing 
x_mix, y_mix, _ = mixup(x_tr_norm, y_tr_norm, c[train_idx], alpha=alpha, seed=seed)

#Define DataSet
train_ds = Training(x_mix,       y_mix,      x_tr_norm)
val_ds   = Validation(x_val_norm, y_val_norm, c_val)
test_ds  = Validation(x_te_norm,  y_te_norm,  c_te)

train_loader, val_loader = get_dataloaders( train_ds, val_ds, batch_size=batch_size, num_workers=0, seed=seed)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0)

model     = Sohoite(input_dim=x.shape[1], n_outputs=n_output).to(device) # Algorithm Function Call 
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2) #AdamW uses an adjusted weighted gradient 
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=30, min_lr=1e-6)

best_val_loss     = float('inf')
best_state        = None
epochs_no_improve = 0
stopped_epoch     = epochs
train_loss_hist   = []
val_loss_hist     = []

print(f'  {"Epoch":>6} | {"Train Loss":>12} | {"Val Loss":>10} |')
print(f'  {"-"*40}')

#training 
for epoch in range(1, epochs + 1):
    tr_loss  = train_epoch(model, train_loader, optimizer, device,
                           x_mean, x_std, y_mean, y_std)
    val_loss, _, _ = evaluate(model, val_loader, device)
    scheduler.step(val_loss)
    train_loss_hist.append(tr_loss)
    val_loss_hist.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss     = val_loss
        best_state        = {k: v.clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
        mark = '*'
    else:
        epochs_no_improve += 1
        mark = ' '

    if epoch % 50 == 0 or epoch == 1:
        print(f'  {epoch:>6} | {tr_loss:>12.6f} | {val_loss:>10.6f} | {mark}')

    if epochs_no_improve >= patience:
        stopped_epoch = epoch
        break

print(f'\n  Stopped : epoch {stopped_epoch}  |  Best val loss : {best_val_loss:.6f}')

#  Evaluate 
model.load_state_dict(best_state)

_, y_pred_val_norm, y_true_val_norm = evaluate(model, val_loader, device)
_, y_pred_te_norm,  y_true_te_norm  = evaluate(model, test_loader, device)

val_results,  val_meanmae  = tester(y_pred_val_norm, y_true_val_norm, y_mean, y_std, target_col)
test_results, test_meanmae = tester(y_pred_te_norm,  y_true_te_norm,   y_mean, y_std, target_col)


y_pred_val_w = (y_pred_val_norm * y_std + y_mean).numpy()
y_true_val_w = (y_true_val_norm * y_std + y_mean).numpy()
y_pred_te_w  = (y_pred_te_norm  * y_std + y_mean).numpy()
y_true_te_w  = (y_true_te_norm  * y_std + y_mean).numpy()

print(f'\n  {"Output":>10} | {"Val MAE":>9} | {"Val R2":>8} | {"Test MAE":>9} | {"Test R2":>8}')
print(f'  {"-"*58}')
for col in target_col:
    print(f'  {col:>10} | {val_results[col]["mae"]:>9.4f} | '
          f'{val_results[col]["r2"]:>8.4f} | '
          f'{test_results[col]["mae"]:>9.4f} | '
          f'{test_results[col]["r2"]:>8.4f}')


   Epoch |   Train Loss |   Val Loss |
  ----------------------------------------
       1 |    73.996153 |   1.663052 | *
      50 |     9.731524 |   0.544991 |  
     100 |     8.763752 |   0.471388 |  
     150 |     8.122203 |   0.474760 |  

  Stopped : epoch 168  |  Best val loss : 0.432685

      Output |   Val MAE |   Val R2 |  Test MAE |  Test R2
  ----------------------------------------------------------
      Ftan_c |   15.8941 |  -0.5546 |   17.1597 |   0.2310
       Power |    4.8152 |   0.8405 |    6.2705 |   0.7604


In [77]:
search_space = {
    'seed'       : [42, 100,123],       
    'alpha'      : [0.2, 0.4,0.5],      
    'dropout'    : [0.0],             
    'Lambda'     : [0.1, 0.2,0.3],      
    'epochs'     : [500],           
    'patience'   : [50],            
    'batch_size' : [16],            
    'hidden'     : [
        [32,  64,  32],
        [64,  128, 64],
        [32,  128, 32],
    ],                              
}
# Total : 2×2×1×2×1×1×1×3 = 24 runs — ~12 min on CPU

# ── Generate all combinations ─────────────────────────────────────────────────
keys   = list(search_space.keys())
values = list(search_space.values())
combos = list(itertools.product(*values))

print(f'  Total combinations : {len(combos)}')
print(f'  Parameters         : {keys}')
print(f'  Starting search...\n')

# ── Results storage ───────────────────────────────────────────────────────────
all_results = []
best_mean_std = float('inf')
best_config   = None
best_metrics  = None

# ── Search loop ───────────────────────────────────────────────────────────────
for run_i, combo in enumerate(combos):
    cfg = dict(zip(keys, combo))

    print(f'  Run {run_i+1:>3}/{len(combos)}  {cfg}')

    # ── Apply parameters ──────────────────────────────────────────────────────
    torch.manual_seed(cfg['seed'])
    np.random.seed(cfg['seed'])

    # Re-split with current seed
    train_idx_r, val_idx_r, test_idx_r = split_data(x, y, c, seed=cfg['seed'])

    # Normalize
    x_tr_r  = x[train_idx_r];  y_tr_r = y[train_idx_r]
    x_val_r = x[val_idx_r];    y_val_r = y[val_idx_r]
    x_te_r  = x[test_idx_r];   y_te_r  = y[test_idx_r]
    c_val_r = c[val_idx_r];    c_te_r  = c[test_idx_r]

    x_mean_r = x_tr_r.mean(dim=0)
    x_std_r  = x_tr_r.std(dim=0).clamp(min=1e-8)
    y_mean_r = y_tr_r.mean(dim=0)
    y_std_r  = y_tr_r.std(dim=0).clamp(min=1e-8)

    x_tr_norm_r  = (x_tr_r  - x_mean_r) / x_std_r
    x_val_norm_r = (x_val_r - x_mean_r) / x_std_r
    x_te_norm_r  = (x_te_r  - x_mean_r) / x_std_r
    y_tr_norm_r  = (y_tr_r  - y_mean_r) / y_std_r
    y_val_norm_r = (y_val_r - y_mean_r) / y_std_r
    y_te_norm_r  = (y_te_r  - y_mean_r) / y_std_r

    # Mixup
    x_mix_r, y_mix_r, _ = mixup(
        x_tr_norm_r, y_tr_norm_r, c[train_idx_r],
        alpha=cfg['alpha'], seed=cfg['seed']
    )

    # Datasets
    train_ds_r = Training(x_mix_r,       y_mix_r,      x_tr_norm_r)
    val_ds_r   = Validation(x_val_norm_r, y_val_norm_r, c_val_r)
    test_ds_r  = Validation(x_te_norm_r,  y_te_norm_r,  c_te_r)

    train_loader_r, val_loader_r = get_dataloaders(
        train_ds_r, val_ds_r,
        batch_size=cfg['batch_size'], num_workers=0, seed=cfg['seed']
    )
    test_loader_r = DataLoader(
        test_ds_r, batch_size=cfg['batch_size'], shuffle=False, num_workers=0
    )

    # Model with current dropout
    model_r = Sohoite(input_dim=x.shape[1], n_outputs=n_output, dropout=cfg['dropout'], hidden=cfg['hidden']).to(device)
    optimizer_r = optim.AdamW(model_r.parameters(), lr=1e-3, weight_decay=1e-2)
    scheduler_r = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_r, mode='min', factor=0.5,
        patience=30, min_lr=1e-6
    )

    # Override Lambda for this run
    Lambda = cfg['Lambda']

    # Training loop
    best_val_r     = float('inf')
    best_state_r   = None
    no_improve_r   = 0
    stopped_r      = cfg['epochs']

    for epoch in range(1, cfg['epochs'] + 1):
        train_epoch(model_r, train_loader_r, optimizer_r, device,
                    x_mean_r, x_std_r, y_mean_r, y_std_r)
        val_loss_r, _, _ = evaluate(model_r, val_loader_r, device)
        scheduler_r.step(val_loss_r)

        if val_loss_r < best_val_r:
            best_val_r   = val_loss_r
            best_state_r = {k: v.clone() for k, v in model_r.state_dict().items()}
            no_improve_r = 0
        else:
            no_improve_r += 1
        if no_improve_r >= cfg['patience']:
            stopped_r = epoch
            break

    # Evaluate
    model_r.load_state_dict(best_state_r)
    _, y_pred_val_r, y_true_val_r = evaluate(model_r, val_loader_r, device)
    _, y_pred_te_r,  y_true_te_r  = evaluate(model_r, test_loader_r, device)

    val_res_r,  _ = tester(y_pred_val_r, y_true_val_r, y_mean_r, y_std_r, target_col)
    test_res_r, _ = tester(y_pred_te_r,  y_true_te_r,  y_mean_r, y_std_r, target_col)

    y_pred_val_wr = (y_pred_val_r * y_std_r + y_mean_r).numpy()
    y_true_val_wr = (y_true_val_r * y_std_r + y_mean_r).numpy()
    y_pred_te_wr  = (y_pred_te_r  * y_std_r + y_mean_r).numpy()
    y_true_te_wr  = (y_true_te_r  * y_std_r + y_mean_r).numpy()

    # Collect row
    row = {
        'run'          : run_i + 1,
        'timestamp'    : datetime.now().strftime('%Y-%m-%d %H:%M'),
        'seed'         : cfg['seed'],
        'alpha'        : cfg['alpha'],
        'hidden' : str(cfg['hidden']),  
        'Lambda'       : cfg['Lambda'],
        'epochs_max'   : cfg['epochs'],
        'patience'     : cfg['patience'],
        'batch_size'   : cfg['batch_size'],
        'stopped_epoch': stopped_r,
        'best_val_loss': round(best_val_r, 6),
        'n_train'      : len(train_idx_r),
        'n_val'        : len(val_idx_r),
        'n_test'       : len(test_idx_r),
        'pct_train'    : round(len(train_idx_r)/len(x)*100, 1),
        'pct_val'      : round(len(val_idx_r)  /len(x)*100, 1),
        'pct_test'     : round(len(test_idx_r) /len(x)*100, 1),
    }

    mean_test_std = 0.0
    for col_i, col in enumerate(target_col):
        for split, y_p, y_t, res in [
            ('val',  y_pred_val_wr, y_true_val_wr, val_res_r),
            ('test', y_pred_te_wr,  y_true_te_wr,  test_res_r),
        ]:
            err = y_p[:, col_i] - y_t[:, col_i]
            row[f'{col}_{split}_mae']     = round(res[col]['mae'],            4)
            row[f'{col}_{split}_r2']      = round(res[col]['r2'],             4)
            row[f'{col}_{split}_bias']    = round(float(err.mean()),          4)
            row[f'{col}_{split}_std']     = round(float(err.std()),           4)
            row[f'{col}_{split}_max_err'] = round(float(np.abs(err).max()),   4)
            if split == 'test':
                mean_test_std += float(err.std())

    mean_test_std /= len(target_col)
    row['mean_test_std'] = round(mean_test_std, 4)
    row['best_run']      = ''   # filled in after loop

    all_results.append(row)

    # Track best
    if mean_test_std < best_mean_std:
        best_mean_std = mean_test_std
        best_config   = cfg.copy()
        best_metrics  = row.copy()

    print(f'         mean_test_std={mean_test_std:.4f}  '
          f'best so far={best_mean_std:.4f}')

# ── Mark best run ─────────────────────────────────────────────────────────────
for row in all_results:
    if row['mean_test_std'] == best_mean_std:
        row['best_run'] = 'BEST'
from openpyxl.styles import PatternFill, Font
# ── Write to Excel ────────────────────────────────────────────────────────────
xlsx_file = 'ML_Parameters2.xlsx'
wb_out    =pxl.Workbook()
ws_out    = wb_out.active
ws_out.title = 'Results'

headers = list(all_results[0].keys())
ws_out.append(headers)

# Style header
header_fill = PatternFill(start_color='1F4E79', end_color='1F4E79', fill_type='solid')
header_font = Font(color='FFFFFF', bold=True)
for cell in ws_out[1]:
    cell.fill = header_fill
    cell.font = header_font

# Style for best run
best_fill = PatternFill(start_color='E2EFDA', end_color='E2EFDA', fill_type='solid')

best_std = min(row['mean_test_std'] for row in all_results)
std_col  = headers.index('mean_test_std')
for excel_row in ws_out.iter_rows(min_row=2):
    if excel_row[std_col].value == best_std:
        for cell in excel_row:
            cell.fill = best_fill
            cell.font = Font(bold=True)

# Auto-width columns
for col in ws_out.columns:
    max_len = max(len(str(cell.value or '')) for cell in col)
    ws_out.column_dimensions[col[0].column_letter].width = max_len + 3

wb_out.save(xlsx_file)


# ── Print best config ─────────────────────────────────────────────────────────
print(f'\n{"="*62}')
print(f'  Search complete  —  {len(combos)} runs')
print(f'{"="*62}')
print(f'  Best config:')
for k, v in best_config.items():
    print(f'    {k:<15} : {v}')
print(f'  Best mean test std : {best_mean_std:.4f}')
print(f'{"="*62}')
print(f'  Saved -> {xlsx_file}')
print(f'{"="*62}')


  Total combinations : 81
  Parameters         : ['seed', 'alpha', 'dropout', 'Lambda', 'epochs', 'patience', 'batch_size', 'hidden']
  Starting search...

  Run   1/81  {'seed': 42, 'alpha': 0.2, 'dropout': 0.0, 'Lambda': 0.1, 'epochs': 500, 'patience': 50, 'batch_size': 16, 'hidden': [32, 64, 32]}
         mean_test_std=4.4834  best so far=4.4834
  Run   2/81  {'seed': 42, 'alpha': 0.2, 'dropout': 0.0, 'Lambda': 0.1, 'epochs': 500, 'patience': 50, 'batch_size': 16, 'hidden': [64, 128, 64]}
         mean_test_std=5.0142  best so far=4.4834
  Run   3/81  {'seed': 42, 'alpha': 0.2, 'dropout': 0.0, 'Lambda': 0.1, 'epochs': 500, 'patience': 50, 'batch_size': 16, 'hidden': [32, 128, 32]}
         mean_test_std=4.6074  best so far=4.4834
  Run   4/81  {'seed': 42, 'alpha': 0.2, 'dropout': 0.0, 'Lambda': 0.2, 'epochs': 500, 'patience': 50, 'batch_size': 16, 'hidden': [32, 64, 32]}
         mean_test_std=5.3664  best so far=4.4834
  Run   5/81  {'seed': 42, 'alpha': 0.2, 'dropout': 0.0, 'Lamb

In [78]:
# ── Write to Excel ────────────────────────────────────────────────────────────
xlsx_file = 'ML_Parameters2.xlsx'
wb_out    =pxl.Workbook()
ws_out    = wb_out.active
ws_out.title = 'Results'

headers = list(all_results[0].keys())
ws_out.append(headers)

# Style header
header_fill = PatternFill(start_color='1F4E79', end_color='1F4E79', fill_type='solid')
header_font = Font(color='FFFFFF', bold=True)
for cell in ws_out[1]:
    cell.fill = header_fill
    cell.font = header_font

# Style for best run
best_fill = PatternFill(start_color='E2EFDA', end_color='E2EFDA', fill_type='solid')

best_std = min(row['mean_test_std'] for row in all_results)
std_col  = headers.index('mean_test_std')
for excel_row in ws_out.iter_rows(min_row=2):
    if excel_row[std_col].value == best_std:
        for cell in excel_row:
            cell.fill = best_fill
            cell.font = Font(bold=True)

# Auto-width columns
for col in ws_out.columns:
    max_len = max(len(str(cell.value or '')) for cell in col)
    ws_out.column_dimensions[col[0].column_letter].width = max_len + 3

wb_out.save(xlsx_file)
